# 360 Object dep tu 1 anh bang TripoSR

Notebook nay dung TripoSR de tao mesh 360 tu mot anh vat the don. Day la huong phu hop hon pipeline depth map neu ban muon xoay quanh object dep.

Anh dau vao nen la mot vat the ro, nen don gian, anh net, anh mat truoc hoac goc 3/4. Phong canh/toan canh khong phu hop voi TripoSR.

In [ ]:
# Kiem tra GPU Colab
!nvidia-smi

In [ ]:
# Cai dat TripoSR. Lan dau co the mat vai phut.
!pip -q install --upgrade pip
!pip -q install "setuptools<82" wheel "jedi>=0.16"
!pip -q install torch torchvision --index-url https://download.pytorch.org/whl/cu124
!git clone https://github.com/VAST-AI-Research/TripoSR.git /content/TripoSR
%cd /content/TripoSR
!pip -q install -r requirements.txt
!pip -q install "numpy==1.26.4" "cupy-cuda12x==13.6.0" onnxruntime trimesh

In [ ]:
# Upload anh vat the don JPG/PNG
from google.colab import files
from pathlib import Path
import os
import shutil

input_dir = Path('/content/triposr_inputs')
output_dir = Path('/content/triposr_outputs')
input_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
os.chdir(str(input_dir))
print('Upload dir:', input_dir)

uploaded = files.upload()
assert uploaded, 'Can upload 1 anh JPG/PNG.'
name = next(iter(uploaded.keys()))
image_path = input_dir / name
with open(image_path, 'wb') as f:
    f.write(uploaded[name])
print('Input:', image_path)


In [ ]:
# Chay TripoSR xuat GLB. chunk-size 4096 nhe hon cho T4; mc-resolution 256 dep hon 128 nhung ton VRAM hon.
%cd /content/TripoSR
!python run.py "$image_path" --output-dir "$output_dir" --model-save-format glb --mc-resolution 256 --chunk-size 4096 --foreground-ratio 0.85

glb_path = output_dir / '0' / 'mesh.glb'
assert glb_path.exists(), 'Khong thay mesh.glb. Hay xem log o cell tren.'
print('GLB:', glb_path)

In [ ]:
# Preview GLB xoay 360 trong Colab
from base64 import b64encode
from IPython.display import HTML, display

with open(glb_path, 'rb') as f:
    glb_b64 = b64encode(f.read()).decode('utf-8')

display(HTML(f'''
<script type="module" src="https://unpkg.com/@google/model-viewer/dist/model-viewer.min.js"></script>
<model-viewer src="data:model/gltf-binary;base64,{glb_b64}" camera-controls auto-rotate shadow-intensity="1" environment-image="neutral" style="width:100%;height:560px;background:#eef2f7;border:1px solid #d8dee8;border-radius:8px;"></model-viewer>
'''))

In [ ]:
# Xuat them OBJ/PLY de tai ve neu can
!pip -q install trimesh
import trimesh

mesh = trimesh.load(str(glb_path), force='mesh')
obj_path = output_dir / 'mesh.obj'
ply_path = output_dir / 'mesh.ply'
mesh.export(str(obj_path))
mesh.export(str(ply_path))

files.download(str(glb_path))
files.download(str(obj_path))
files.download(str(ply_path))